# Genuine grokking on atomic Cayley tables: $S_5,\ldots,S_8$

Каждая перестановка здесь является непрозрачным ID, а target — ID произведения. Это исходная постановка успешного эксперимента $S_5$ без coordinate-wise shortcut. $S_3$ и $S_4$ исключены из основной серии как слишком маленькие задачи.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys, torch
assert torch.cuda.is_available(), 'Выберите GPU runtime'
REPO='/content/2026-Project-202'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/intsystems/2026-Project-202.git',REPO],check=True)
WORKDIR=os.path.join(REPO,'code','Grokking')
os.chdir(WORKDIR)
# Перед запуском загрузите sn_atomic_grokking_colab.py в Files Colab.
assert os.path.exists('/content/sn_atomic_grokking_colab.py')
shutil.copy2('/content/sn_atomic_grokking_colab.py', os.path.join(WORKDIR,'sn_atomic_grokking_colab.py'))
subprocess.run([sys.executable,'-m','pip','install','-q','einops','pandas','tqdm'],check=True)

In [ ]:
import importlib, sn_atomic_grokking_colab
importlib.reload(sn_atomic_grokking_colab)
from sn_atomic_grokking_colab import AtomicConfig, run_atomic_sweep

# Первый осмысленный pilot: S_5 и S_6, одна и та же архитектура и AdamW.
PILOT = AtomicConfig(
    output_root='/content/drive/MyDrive/grokking_sn_atomic',
    protocol_name='atomic_cayley_v1',
    n_values=(5,6), seeds=(42,),
    train_fraction=0.5,
    sampled_pairs_by_n={5:14_400, 6:100_000, 7:200_000, 8:300_000},
    learning_rate=1e-3, weight_decay=0.2, betas=(0.9,0.98),
    d_model=128, d_mlp=512, d_head=32, n_heads=4,
    required_gap_steps=10_000, max_steps=300_000,
    log_every=20, checkpoint_every=1_000,
)
run_atomic_sweep(PILOT)

In [ ]:
from pathlib import Path
import json
for path in sorted(Path(PILOT.output_root).glob('atomic_cayley_v1/S_*/seed_*/COMPLETED.json')):
    print(path.parent, json.loads(path.read_text()))

Если $S_5$ воспроизводит gap, а $S_6$ не генерализуется, сначала увеличьте только `sampled_pairs_by_n[6]` до 200000. Не меняйте архитектуру и AdamW отдельно для разных $n$. После pilot добавляйте $S_7$, затем $S_8$.

In [ ]:
# Финальный запуск после успешного pilot, лучше на новых seeds.
FULL = AtomicConfig(
    output_root=PILOT.output_root, protocol_name='atomic_cayley_v1_final',
    n_values=(5,6,7,8), seeds=(43,44,45),
    train_fraction=0.5,
    sampled_pairs_by_n={5:14_400, 6:100_000, 7:200_000, 8:300_000},
    learning_rate=1e-3, weight_decay=0.2, betas=(0.9,0.98),
    d_model=128, d_mlp=512, d_head=32, n_heads=4,
    required_gap_steps=10_000, max_steps=300_000,
)
# run_atomic_sweep(FULL)  # раскомментировать после pilot
